# 03 — Entraîner Sharm Image Base pour les compositions

**Objectif :** suivre les données, les pertes, les paramètres et les checkpoints du projecteur spatial.

~~~text
3 600 compositions RGBA → features 8B / 4B et latents
                       → trajectoires 8B
                       → warmup embeddings, 1 000 pas
                       → flow supervisé + distillation, 1 500 pas maximum
                       → meilleur candidat BF16 → galerie appariée
~~~

Le profil distant conserve le dataset complet et le nombre de pas. Son arrêt anticipé préexistant est défini dans le trainer ; aucune réduction manuelle n'est nécessaire.

In [ ]:
from pathlib import Path
import sys

book_dir = next(
    folder
    for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent / "notebooks", parent / "2-LoRA-qwen/notebooks", parent)
    if (folder / "notebook_utils.py").is_file()
)
if str(book_dir) not in sys.path:
    sys.path.insert(0, str(book_dir))
from notebook_utils import (
    PROJECT, REPO, ARTIFACTS, read_json, read_jsonl, sha256,
    contained, current_dataset, archived_report, result_file,
    show_json, table, show_image, run_script,
)
print("Projet :", PROJECT.name)


In [ ]:
profile = read_json(PROJECT / "03-spatial-distillation/configs/remote16gb-256.json")
show_json(PROJECT / "03-spatial-distillation/configs/remote16gb-256.json")

## Ce qui est optimisé

- **Warmup :** alignement des embeddings et contexte entre tokens.
- **Flow :** prédiction du bruit/flow sur les images de composition, trajectoires 8B et maintien de l'alignement des embeddings.
- **Boîtes :** facultatives en inférence ; leur dropout évite de les rendre obligatoires pour chaque prompt.
- **Profil courant :** rang LoRA du diffuseur à 0 ; seul le projecteur reçoit des gradients. Le code permet un LoRA de diffusion dans d'autres profils.

Un checkpoint contient aussi l'état d'optimiseur et l'identité du run pour reprendre. Le meilleur candidat exportable est choisi sur la validation numérique. Sa qualité visuelle reste à examiner.

In [ ]:
archive = archived_report()
if archive:
    for phase in ("warmup", "flow"):
        print(phase)
        show_json(result_file(f"run/{phase}/training-complete.json"))
else:
    observation = read_json(ARTIFACTS / "spatial/remote-observation.json", {})
    table([{
        "relevé UTC": observation.get("checked_at", "absent"),
        "phase": observation.get("run", {}).get("stage", "absent"),
        "pas": observation.get("flow", {}).get("step", "absent"),
        "objectif": observation.get("flow", {}).get("steps", "absent"),
        "disque libre Gio": observation.get("free_disk_gib", "absent"),
    }], ["relevé UTC", "phase", "pas", "objectif", "disque libre Gio"])
    print("Ceci est un relevé daté, pas une interrogation du serveur en direct.")

In [ ]:
history = read_jsonl(result_file("run/flow/history.jsonl"))
table([{
    "pas": r["step"],
    "validation numérique": r.get("numerical_validation_loss"),
    "temps s": round(r.get("seconds", 0), 1),
    "pic GPU Gio": round(r.get("gpu_peak_gib", 0), 3),
} for r in history], ["pas", "validation numérique", "temps s", "pic GPU Gio"])

## Courbe de validation

La courbe utilise uniquement les points réellement enregistrés. Matplotlib est facultatif pour la lecture : si le kernel ne le possède pas, le tableau précédent reste disponible. Ne pas comparer directement cette perte composite à la MSE relative historique des embeddings.

In [ ]:
import importlib.util

if history and importlib.util.find_spec("matplotlib"):
    import matplotlib.pyplot as plt
    points = [r for r in history if "numerical_validation_loss" in r]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot([r["step"] for r in points], [r["numerical_validation_loss"] for r in points], marker="o")
    ax.set(xlabel="Pas de flow", ylabel="Perte de validation numérique", title="Mesures enregistrées")
    ax.grid(alpha=0.25)
    plt.show()
elif not history:
    print("Historique à rapatrier après le calcul distant.")
else:
    print("Matplotlib absent : consulter le tableau des mesures.")

## Reproduire sur le serveur après libération du GPU

La cellule suivante affiche la commande. Elle ne lance rien tant que le drapeau reste à False. Pour une reproduction, utiliser le kernel CUDA du workspace Linux avec ses poids aux révisions figées et `SHARM_NOTEBOOK_GPU=1`. Le helper refuse une seconde chaîne active.

Le démarrage complet d'un workspace neuf, les étapes de téléchargement séquentielles et la reprise sont décrits dans [REMOTE.md](../03-spatial-distillation/REMOTE.md). Le notebook appelle le même runner ; il ne déplace ni ne modifie le run actif.

In [ ]:
RUN_SPATIAL_PIPELINE = False
run_script("03-spatial-distillation/run_remote.py",
           enabled=RUN_SPATIAL_PIPELINE, gpu=True)

**Suite :** [04 — Générations et évaluation](04_generations_et_evaluation.ipynb).